# Exploratory Data Analysis (EDA) — Roller Coaster Dataset

**Project objective:** Analyze a real-world roller coaster dataset to uncover patterns, trends, correlations, and key influencing factors.

**Dataset:** `coaster_db.csv` (1,087 rows × 56 columns)

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

This notebook follows the workflow demonstrated in Rob Mulla's EDA tutorial: data understanding → data preparation → feature understanding → feature relationships → questions and insights.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
pd.set_option("display.max_columns", 200)

# Load dataset
df = pd.read_csv("coaster_db.csv")

print(f"Dataset shape: {df.shape}")


## 1. Data Understanding

In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.tail())
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nStatistical summary:")
display(df.describe(include="all").T)


## 2. Data Preparation

In [ ]:
cols = ["coaster_name", "Location", "Status", "Type", "Manufacturer", "year_introduced", "latitude", "longitude", "Type_Main", "opening_date_clean", "speed_mph", "height_ft", "Inversions_clean", "Gforce_clean"]
data = df[cols].copy()
data = data.rename(columns={"coaster_name": "Coaster_Name", "year_introduced": "Year_Introduced", "Type_Main": "Type_Main", "opening_date_clean": "Opening_Date", "speed_mph": "Speed_MPH", "height_ft": "Height_FT", "Inversions_clean": "Inversions", "Gforce_clean": "Gforce"})
data["Opening_Date"] = pd.to_datetime(data["Opening_Date"], errors="coerce")
for c in ["Year_Introduced", "Speed_MPH", "Height_FT", "Inversions", "Gforce"]:
    data[c] = pd.to_numeric(data[c], errors="coerce")
print("Missing values:")
display(data.isna().sum().sort_values(ascending=False))
print("Duplicate rows:", data.duplicated().sum())
print("Duplicate coaster/location/date combinations:", data.duplicated(subset=["Coaster_Name", "Location", "Opening_Date"]).sum())
data = data.drop_duplicates(subset=["Coaster_Name", "Location", "Opening_Date"]).reset_index(drop=True)
print("Shape after duplicate removal:", data.shape)
analysis = data.dropna(subset=["Speed_MPH", "Height_FT", "Gforce", "Inversions"]).copy()
print("Rows available for complete numerical relationship analysis:", len(analysis))


## 3. Feature Understanding

In [ ]:
year_counts = data["Year_Introduced"].value_counts().head(10).sort_values()
plt.figure(figsize=(10, 6))
year_counts.plot(kind="barh")
plt.title("Top 10 Years with the Most Roller Coasters Introduced")
plt.xlabel("Number of Coasters")
plt.ylabel("Year Introduced")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(data["Speed_MPH"].dropna(), bins=25, kde=True)
plt.title("Distribution of Roller Coaster Speed")
plt.xlabel("Speed (MPH)")
plt.ylabel("Number of Coasters")
plt.tight_layout()
plt.show()

top_manufacturers = data["Manufacturer"].value_counts().head(10).sort_values()
plt.figure(figsize=(10, 6))
top_manufacturers.plot(kind="barh")
plt.title("Top 10 Roller Coaster Manufacturers")
plt.xlabel("Number of Coasters")
plt.ylabel("Manufacturer")
plt.tight_layout()
plt.show()


## 4. Feature Relationships

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=data, x="Height_FT", y="Speed_MPH", hue="Type_Main", alpha=0.7)
plt.title("Relationship Between Height and Speed")
plt.xlabel("Height (ft)")
plt.ylabel("Speed (MPH)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=data, x="Year_Introduced", y="Speed_MPH", hue="Type_Main", alpha=0.6)
plt.title("Roller Coaster Speed by Year Introduced")
plt.xlabel("Year Introduced")
plt.ylabel("Speed (MPH)")
plt.tight_layout()
plt.show()

corr_cols = ["Year_Introduced", "Speed_MPH", "Height_FT", "Inversions", "Gforce"]
corr = data[corr_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()
display(corr)


## 5. Key Questions

In [ ]:
q1 = (data[data["Location"].ne("Other")].groupby("Location")["Speed_MPH"].agg(["mean", "count"]).query("count >= 10").sort_values("mean", ascending=False).head(10))
q1 = q1.rename(columns={"mean":"Mean_Speed_MPH", "count":"Coaster_Count"})
display(q1)
plt.figure(figsize=(10, 6))
q1["Mean_Speed_MPH"].sort_values().plot(kind="barh")
plt.title("Top Locations by Average Roller Coaster Speed")
plt.xlabel("Average Speed (MPH)")
plt.ylabel("Location")
plt.tight_layout()
plt.show()

q2 = (data[(data["Type_Main"] == "Steel") & data["Location"].ne("Other")].groupby("Location")["Coaster_Name"].count().sort_values(ascending=False).head(10))
display(q2)
plt.figure(figsize=(10, 6))
q2.sort_values().plot(kind="barh")
plt.title("Top Locations by Number of Steel Roller Coasters")
plt.xlabel("Number of Steel Coasters")
plt.ylabel("Location")
plt.tight_layout()
plt.show()

q3 = (data[data["Type_Main"] == "Steel"].groupby("Year_Introduced")["Coaster_Name"].count().sort_values(ascending=False).head(10))
display(q3)

type_speed = data.groupby("Type_Main")["Speed_MPH"].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)
display(type_speed)
plt.figure(figsize=(8, 5))
type_speed["mean"].plot(kind="bar")
plt.title("Average Speed by Coaster Type")
plt.xlabel("Coaster Type")
plt.ylabel("Average Speed (MPH)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Findings and Conclusion

### Main findings
1. The dataset contains **1,087 roller-coaster records and 56 columns** before preparation.
2. Duplicate coaster records are present; removing repeated `(Coaster_Name, Location, Opening_Date)` combinations prevents repeated rides from distorting summaries.
3. Roller-coaster construction became much more concentrated in the **late 1990s and early 2000s**.
4. Most recorded coaster speeds fall roughly in the **35–60 MPH** range, with about **50 MPH** appearing frequently.
5. **Steel coasters generally show higher speed and height** than wooden coasters.
6. Speed and height have a **moderately positive relationship** in the cleaned analysis.
7. Speed and G-force show a **weak positive relationship**, so speed alone does not explain G-force.
8. Inversions are strongly associated with coaster design/type, and the distribution differs between wood and steel coasters.

### Business/analytical takeaway
The strongest influencing factors visible in the EDA are **coaster type, height, and introduction period**. The analysis also demonstrates why missing-value handling and duplicate detection are important before interpreting correlations.

> **Important:** Correlation identifies association, not causation.


## 7. Skills Demonstrated

- Data loading with Pandas
- Dataset profiling and statistical summaries
- Missing-value and duplicate inspection
- Data type conversion
- Feature selection
- Group-by aggregation
- Histograms, bar charts, scatter plots and heatmaps
- Correlation analysis
- Insight generation and structured reporting
